# 02 Preprocessing
Feature engineering, encoding, scaling, feature selection, and class imbalance handling for churn modeling and revenue forecasting.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from src.data_loader import load_telco_data, split_train_val_test
from src.preprocessing import (
    create_derived_features,
    encode_label_series,
    feature_importance_tree,
    feature_selection_correlation,
    feature_selection_mutual_information,
    feature_selection_rfe,
    fit_feature_preprocessor,
    handle_class_imbalance,
    handle_missing_values,
    prepare_features,
    prepare_train_val_test_features,
    treat_outliers_iqr,
    transform_features,
)
from src.utils import PROJECT_ROOT, ensure_directory, save_pickle, set_random_seed

set_random_seed(42)
processed_dir = ensure_directory(PROJECT_ROOT / 'data' / 'processed')
models_dir = ensure_directory(PROJECT_ROOT / 'models')

frame = load_telco_data()
frame = handle_missing_values(frame)
frame = create_derived_features(frame)
frame = treat_outliers_iqr(frame, numeric_columns=['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'ServiceCount', 'ContractValue'])

print('Label encoding is useful for binary or ordered fields; one-hot encoding is better for nominal categories.')
encoded_contract, contract_encoder = encode_label_series(frame['Contract'])
one_hot_internet = pd.get_dummies(frame[['InternetService']], drop_first=False)
print('Contract label mapping:', dict(zip(contract_encoder.classes_, contract_encoder.transform(contract_encoder.classes_))))
print('One-hot columns:', one_hot_internet.columns.tolist())

numeric_demo = frame[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
standard_scaled = StandardScaler().fit_transform(numeric_demo)
minmax_scaled = MinMaxScaler().fit_transform(numeric_demo)
print('StandardScaler summary: mean =', np.round(standard_scaled.mean(axis=0), 3), 'std =', np.round(standard_scaled.std(axis=0), 3))
print('MinMaxScaler summary: min =', np.round(minmax_scaled.min(axis=0), 3), 'max =', np.round(minmax_scaled.max(axis=0), 3))

train_frame, validation_frame, test_frame = split_train_val_test(frame, target_column='Churn', stratify=True, random_state=42)
X_train, y_train, X_validation, y_validation, X_test, y_test, preprocessor, feature_names = prepare_train_val_test_features(
    train_frame,
    validation_frame,
    test_frame,
    target_column='Churn',
    drop_columns=['customerID'],
    scaler='standard',
)

print('Split sizes:')
print({'train': X_train.shape, 'validation': X_validation.shape, 'test': X_test.shape})
print('Feature count:', len(feature_names))

X_full, y_full, _, full_feature_names = prepare_features(frame, target_column='Churn', drop_columns=['customerID'], scaler='standard')
print('Correlation-based feature filter count:', len(feature_selection_correlation(X_full, threshold=0.85)))
print('Top mutual information features:')
print(feature_selection_mutual_information(X_full, y_full, task='classification').head(10))
print('RFE-selected features:')
print(feature_selection_rfe(X_full, y_full, task='classification', n_features_to_select=15))
print('Tree feature importance top 10:')
print(feature_importance_tree(X_full, y_full, task='classification').head(10))

X_smote, y_smote = handle_class_imbalance(X_train, y_train, method='smote', random_state=42)
X_under, y_under = handle_class_imbalance(X_train, y_train, method='undersample', random_state=42)
print('Original training distribution:')
print(y_train.value_counts(normalize=True).round(3))
print('SMOTE training distribution:')
print(y_smote.value_counts(normalize=True).round(3))
print('Undersampled training distribution:')
print(y_under.value_counts(normalize=True).round(3))

processed_train_path = processed_dir / 'churn_train_features.csv'
processed_validation_path = processed_dir / 'churn_validation_features.csv'
processed_test_path = processed_dir / 'churn_test_features.csv'
X_train.assign(Churn=y_train).to_csv(processed_train_path, index=False)
X_validation.assign(Churn=y_validation).to_csv(processed_validation_path, index=False)
X_test.assign(Churn=y_test).to_csv(processed_test_path, index=False)

categorical_encoder = preprocessor.named_transformers_['categorical'].named_steps['encoder']
numeric_scaler = preprocessor.named_transformers_['numeric'].named_steps['scaler']
save_pickle(categorical_encoder, models_dir / 'encoder.pkl')
save_pickle(numeric_scaler, models_dir / 'scaler.pkl')

print('Processed data saved to data/processed/.')